# Inter-Patient TCN: RevIN vs No-RevIN Comparison

**Author:** Ibrahim Hanafy  
**Date:** August 2026  

**Goal:** Train a **single shared TCN** on pooled train patients, evaluate on **unseen** test patients.  
Compare with and without RevIN — RevIN should help most here due to inter-patient distribution shift.

**All MIT-BIH patients** — auto-discovered, randomly split 75/25 (train/test).

**Hyperparameters (MOGWO Pareto-best, Eval 28):**

| Parameter | Value | Source |
|-----------|-------|--------|
| `n_filters` | 128 | MOGWO Pareto-best (RMSE) |
| `dropout` | 0.131 | MOGWO Pareto-best |
| `lr` | 0.00451 | MOGWO Pareto-best |
| `n_blocks` | 3 | Fixed (RF=29 ≫ lookback=10) |
| `kernel_size` | 3 | Fixed (Bai et al.) |
| `batch_size` | 128 | Fixed |
| `lookback` | 10 | Fixed (Dudukcu et al.) |

**Reference:** Kim et al. (2022) — Reversible Instance Normalization for Accurate Time-Series Forecasting against Distribution Shift (ICLR 2022)

---

## 1 — Imports & Setup

In [ ]:
!pip install wfdb -q

In [ ]:
import os, time, gc, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import wilcoxon
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, Add, Activation, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f'TensorFlow {tf.__version__}')
print(f'GPUs: {tf.config.list_physical_devices("GPU")}')

## 2 — Configuration

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
DATA_DIR = r"/kaggle/input/datasets/rracer17/mit-bih-mitdb/mit-bih-arrhythmia-database-1.0.0"

FS           = 360
TOTAL_STEPS  = 100_000
TRAIN_STEPS  = 40_000
VAL_STEPS    = 10_000
TEST_STEPS   = 50_000
LOOKBACK     = 10
HORIZON      = 10

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Auto-discover ALL patients ────────────────────────────────────────────────
ALL_PATIENTS = sorted(set(
    os.path.splitext(os.path.basename(f))[0]
    for f in glob.glob(os.path.join(DATA_DIR, '*.hea'))
))
print(f'Discovered {len(ALL_PATIENTS)} patients: {ALL_PATIENTS}')

# ── Random 75/25 train/test split ─────────────────────────────────────────────
n_test = max(2, int(len(ALL_PATIENTS) * 0.25))
shuffled = np.random.permutation(ALL_PATIENTS).tolist()
TEST_PATIENTS  = sorted(shuffled[:n_test])
TRAIN_PATIENTS = sorted(shuffled[n_test:])

print(f'\nTrain patients ({len(TRAIN_PATIENTS)}): {TRAIN_PATIENTS}')
print(f'Test  patients ({len(TEST_PATIENTS)}):  {TEST_PATIENTS}')

# ── 10 random test patients for signal visualization ──────────────────────────
VIZ_PATIENTS = sorted(np.random.choice(
    TEST_PATIENTS, size=min(10, len(TEST_PATIENTS)), replace=False).tolist())

# ── MOGWO-Optimized Hyperparameters ───────────────────────────────────────────
NUM_FILTERS   = 128    # MOGWO Pareto-best
DROPOUT_RATE  = 0.131  # MOGWO Pareto-best
LEARNING_RATE = 0.00451  # MOGWO Pareto-best
NUM_BLOCKS    = 3      # RF = 29 ≫ lookback=10
KERNEL_SIZE   = 3
BATCH_SIZE    = 128

# ── Training ─────────────────────────────────────────────────────────────────
EPOCHS   = 200
PATIENCE = 50

# ── Output ───────────────────────────────────────────────────────────────────
OUT_DIR = '/kaggle/working'
PLOT_DIR = os.path.join(OUT_DIR, 'plots_inter_revin')
os.makedirs(PLOT_DIR, exist_ok=True)

print(f'\nInter-Patient TCN RevIN Comparison')
print(f'Viz sample  : {len(VIZ_PATIENTS)} → {VIZ_PATIENTS}')
print(f'Horizon     : H={HORIZON}, Lookback={LOOKBACK}')
print(f'TCN         : blocks={NUM_BLOCKS} (RF=29), filters={NUM_FILTERS}, kernel={KERNEL_SIZE}')
print(f'Dropout={DROPOUT_RATE}, LR={LEARNING_RATE}, Batch={BATCH_SIZE}')
print(f'Training    : epochs={EPOCHS}, patience={PATIENCE}')

## 3 — Data Pipeline

**Inter-patient key difference:** Pool all train patients into one dataset.  
Global MinMax scaler fit on pooled train data only — test patients transformed with same scaler.  
This simulates the real scenario: no access to test patient statistics at train time.

In [ ]:
def load_ecg_signal(record_id, data_dir, n_steps=100_000):
    """Load MLII lead from MIT-BIH, truncate/pad to n_steps."""
    path = os.path.join(data_dir, record_id)
    rec  = wfdb.rdrecord(path)
    sig_names_upper = [s.upper() for s in rec.sig_name]
    ch = sig_names_upper.index('MLII') if 'MLII' in sig_names_upper else 0
    signal = rec.p_signal[:, ch].astype(np.float32)
    if len(signal) < n_steps:
        pad = np.full(n_steps - len(signal), signal[-1], dtype=np.float32)
        signal = np.concatenate([signal, pad])
    return signal[:n_steps]


def make_multistep_sequences(signal, lookback, horizon):
    """X = lookback window, y = next H steps (Multi-Output)."""
    X, y = [], []
    for i in range(len(signal) - lookback - horizon + 1):
        X.append(signal[i : i + lookback])
        y.append(signal[i + lookback : i + lookback + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def compute_metrics(y_true, y_pred):
    yt, yp = y_true.flatten(), y_pred.flatten()
    return {
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'MAE':  float(mean_absolute_error(yt, yp)),
        'R2':   float(r2_score(yt, yp)),
    }


print('Data functions defined.')

In [ ]:
# ── Load all raw signals ──────────────────────────────────────────────────────
raw_signals = {}
print(f'Loading {len(ALL_PATIENTS)} patients...\n')
for rid in tqdm(ALL_PATIENTS, desc='Loading'):
    raw_signals[rid] = load_ecg_signal(rid, DATA_DIR, TOTAL_STEPS)
    tqdm.write(f'  Patient {rid:>3s} | samples={len(raw_signals[rid]):,}')

# ── Signal statistics ─────────────────────────────────────────────────────────
print(f'\n--- Signal Statistics ---')
print(f'{"":>6} {"Patient":>8} {"Mean":>10} {"Std":>10} {"Min":>10} {"Max":>10}')
for rid in ALL_PATIENTS:
    s = raw_signals[rid]
    split = 'TRAIN' if rid in TRAIN_PATIENTS else 'TEST '
    print(f'  {split} {rid:>3s} {s.mean():>10.2f} {s.std():>10.2f} {s.min():>10.2f} {s.max():>10.2f}')

print(f'\nLoaded {len(raw_signals)} patients.')

In [ ]:
# ── Build inter-patient datasets ──────────────────────────────────────────────
# Pool train patients, global scaler fit on train only

# 1. Pool train signals
train_pool = np.concatenate([raw_signals[rid] for rid in TRAIN_PATIENTS])
print(f'Pooled train signal: {len(train_pool):,} samples from {len(TRAIN_PATIENTS)} patients')

# 2. Fit global scaler on train pool
global_scaler = MinMaxScaler(feature_range=(0, 1))
train_pool_norm = global_scaler.fit_transform(train_pool.reshape(-1, 1)).flatten()

# 3. Create train sequences from pooled data
X_train_all, y_train_all = make_multistep_sequences(train_pool_norm, LOOKBACK, HORIZON)
print(f'Train sequences: X={X_train_all.shape}, y={y_train_all.shape}')

# 4. Train/val split (80/20 of pooled sequences)
n_train = int(len(X_train_all) * 0.8)
indices = np.random.permutation(len(X_train_all))
train_idx, val_idx = indices[:n_train], indices[n_train:]

X_train = X_train_all[train_idx]
y_train = y_train_all[train_idx]
X_val   = X_train_all[val_idx]
y_val   = y_train_all[val_idx]

print(f'Train: {X_train.shape[0]:,} sequences')
print(f'Val:   {X_val.shape[0]:,} sequences')

# 5. Prepare test data per patient
test_data = {}
for rid in TEST_PATIENTS:
    sig_norm = global_scaler.transform(raw_signals[rid].reshape(-1, 1)).flatten()
    X_te, y_te = make_multistep_sequences(sig_norm, LOOKBACK, HORIZON)
    test_data[rid] = (X_te, y_te)
    print(f'Test patient {rid}: {X_te.shape[0]:,} sequences')

# Reshape for Conv1D
X_train_r = X_train.reshape(-1, LOOKBACK, 1)
X_val_r   = X_val.reshape(-1, LOOKBACK, 1)

# Free memory
del train_pool, train_pool_norm, X_train_all, y_train_all
gc.collect()

print(f'\nDatasets ready. Train={X_train_r.shape}, Val={X_val_r.shape}')

## 4 — Model Architecture

Same TCN + RevIN architecture as intra-patient notebook.  
**Key insight:** RevIN normalizes per-window, so each test sample gets its own stats —  
the model never needs to learn patient-specific amplitude/offset.

In [ ]:
def residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(x)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(out)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1)(x)
    return Add()([x, out])


def build_tcn(lookback, output_size, n_blocks, n_filters, kernel_size,
              dropout_rate, learning_rate):
    """Standard TCN (no RevIN)."""
    inp = Input(shape=(lookback, 1))
    x = inp
    for i in range(n_blocks):
        x = residual_block(x, n_filters, kernel_size, 2 ** i, dropout_rate)
    x = x[:, -1, :]
    x = Dense(n_filters, activation='relu')(x)
    out = Dense(output_size)(x)
    model = Model(inp, out, name=f'TCN_H{output_size}')
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss='mse')
    return model


print('Standard TCN defined.')

In [ ]:
class TCNWithRevIN(tf.keras.Model):
    """TCN wrapped with Reversible Instance Normalization.

    Normalizes each input window by its own mean/std before the TCN,
    then denormalizes the output back to the original scale.
    """

    def __init__(self, lookback, output_size, n_blocks, n_filters,
                 kernel_size, dropout_rate, learning_rate, eps=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.eps = eps
        self.lookback = lookback
        self.horizon  = output_size
        self._lr = learning_rate

        # Learnable affine parameters
        self.revin_gamma = self.add_weight(
            name='revin_gamma', shape=(1, 1, 1),
            initializer='ones', trainable=True)
        self.revin_beta = self.add_weight(
            name='revin_beta', shape=(1, 1, 1),
            initializer='zeros', trainable=True)

        # TCN backbone
        self.backbone = self._build_backbone(
            lookback, output_size, n_blocks, n_filters,
            kernel_size, dropout_rate)

    def _build_backbone(self, lookback, output_size, n_blocks, n_filters,
                        kernel_size, dropout_rate):
        inp = Input(shape=(lookback, 1))
        x = inp
        for i in range(n_blocks):
            x = residual_block(x, n_filters, kernel_size, 2 ** i, dropout_rate)
        x = x[:, -1, :]
        x = Dense(n_filters, activation='relu')(x)
        out = Dense(output_size)(x)
        return Model(inp, out, name='TCN_backbone')

    def call(self, x, training=False):
        # x shape: (batch, lookback, 1)
        # ── RevIN: normalize ──────────────────────────────────────────────
        mean = tf.reduce_mean(x, axis=1, keepdims=True)
        std  = tf.math.reduce_std(x, axis=1, keepdims=True) + self.eps
        x_norm = (x - mean) / std * self.revin_gamma + self.revin_beta

        # ── TCN backbone ──────────────────────────────────────────────────
        out = self.backbone(x_norm, training=training)

        # ── RevIN: denormalize ────────────────────────────────────────────
        mean_sq  = tf.squeeze(mean, axis=[1, 2])
        std_sq   = tf.squeeze(std, axis=[1, 2])
        gamma_sq = tf.squeeze(self.revin_gamma)
        beta_sq  = tf.squeeze(self.revin_beta)

        out = ((out - beta_sq) / (gamma_sq + self.eps)
               * tf.expand_dims(std_sq, 1)
               + tf.expand_dims(mean_sq, 1))
        return out


def build_tcn_revin(lookback, output_size, n_blocks, n_filters,
                    kernel_size, dropout_rate, learning_rate):
    """Build and compile TCN+RevIN model."""
    model = TCNWithRevIN(
        lookback, output_size, n_blocks, n_filters,
        kernel_size, dropout_rate, learning_rate,
        name='TCN_RevIN')
    _ = model(tf.zeros((1, lookback, 1)))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss='mse')
    return model


# Sanity check
_plain = build_tcn(LOOKBACK, HORIZON, NUM_BLOCKS, NUM_FILTERS, KERNEL_SIZE,
                   DROPOUT_RATE, LEARNING_RATE)
_revin = build_tcn_revin(LOOKBACK, HORIZON, NUM_BLOCKS, NUM_FILTERS, KERNEL_SIZE,
                         DROPOUT_RATE, LEARNING_RATE)
print(f'TCN (no RevIN): {_plain.count_params():,} params')
print(f'TCN + RevIN:    {sum(np.prod(v.shape) for v in _revin.trainable_variables):,} params (+2 affine)')
del _plain, _revin
tf.keras.backend.clear_session()

## 5 — Training Helper

**Difference from intra-patient:** One shared model trained on pooled data,  
then evaluated on each unseen test patient separately.

In [ ]:
def train_shared_model(use_revin=False):
    """Train one shared TCN on pooled train data. Return model + history."""
    tf.keras.backend.clear_session()
    gc.collect()
    tf.random.set_seed(SEED)
    np.random.seed(SEED)

    if use_revin:
        model = build_tcn_revin(LOOKBACK, HORIZON, NUM_BLOCKS, NUM_FILTERS,
                                KERNEL_SIZE, DROPOUT_RATE, LEARNING_RATE)
        tag = 'TCN+RevIN'
    else:
        model = build_tcn(LOOKBACK, HORIZON, NUM_BLOCKS, NUM_FILTERS,
                          KERNEL_SIZE, DROPOUT_RATE, LEARNING_RATE)
        tag = 'TCN'

    print(f'\nTraining {tag} on {X_train_r.shape[0]:,} pooled sequences...')

    t0 = time.time()
    history = model.fit(
        X_train_r, y_train,
        validation_data=(X_val_r, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=PATIENCE,
                          restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                              patience=25, min_lr=1e-6),
        ],
        verbose=1
    )
    train_time = time.time() - t0

    ep_trained = len(history.history['loss'])
    best_val = min(history.history['val_loss'])
    print(f'{tag} trained: {ep_trained} epochs, {train_time:.1f}s, best val_loss={best_val:.6f}')

    return model, history, train_time


def evaluate_on_test_patients(model, tag=''):
    """Evaluate trained model on each unseen test patient."""
    results = []
    predictions = {}

    for rid in TEST_PATIENTS:
        X_te, y_te = test_data[rid]
        X_te_r = X_te.reshape(-1, LOOKBACK, 1)

        t0 = time.time()
        preds = model.predict(X_te_r, batch_size=2048, verbose=0)
        pred_time = time.time() - t0

        m = compute_metrics(y_te, preds)
        m['Patient'] = rid
        m['Pred_Time_s'] = round(pred_time, 2)
        results.append(m)
        predictions[rid] = (y_te, preds)

        print(f'  {tag} Patient {rid} | '
              f'RMSE={m["RMSE"]:.6f}  MAE={m["MAE"]:.6f}  '
              f'R²={m["R2"]:.4f}  ({pred_time:.2f}s)')

    return pd.DataFrame(results), predictions


print('Training and evaluation functions defined.')

---
# PART A — No RevIN (Standard TCN)
---

In [ ]:
print('=' * 80)
print('  PART A: Inter-Patient TCN (No RevIN)')
print('=' * 80)

model_nr, hist_nr, time_nr = train_shared_model(use_revin=False)

print(f'\nEvaluating on {len(TEST_PATIENTS)} unseen test patients:\n')
df_no_revin, preds_no_revin = evaluate_on_test_patients(model_nr, tag='NoRevIN')

print(f'\n--- No RevIN Summary ---')
print(f'Mean RMSE : {df_no_revin.RMSE.mean():.6f} ± {df_no_revin.RMSE.std():.6f}')
print(f'Mean MAE  : {df_no_revin.MAE.mean():.6f} ± {df_no_revin.MAE.std():.6f}')
print(f'Mean R²   : {df_no_revin.R2.mean():.4f} ± {df_no_revin.R2.std():.4f}')
print(f'Train time: {time_nr:.1f}s')

del model_nr
gc.collect()

---
# PART B — With RevIN (TCN + RevIN)
---

In [ ]:
print('=' * 80)
print('  PART B: Inter-Patient TCN + RevIN')
print('=' * 80)

model_rv, hist_rv, time_rv = train_shared_model(use_revin=True)

print(f'\nEvaluating on {len(TEST_PATIENTS)} unseen test patients:\n')
df_revin, preds_revin = evaluate_on_test_patients(model_rv, tag='RevIN')

print(f'\n--- RevIN Summary ---')
print(f'Mean RMSE : {df_revin.RMSE.mean():.6f} ± {df_revin.RMSE.std():.6f}')
print(f'Mean MAE  : {df_revin.MAE.mean():.6f} ± {df_revin.MAE.std():.6f}')
print(f'Mean R²   : {df_revin.R2.mean():.4f} ± {df_revin.R2.std():.4f}')
print(f'Train time: {time_rv:.1f}s')

del model_rv
gc.collect()

---
# PART C — Comparison & Visualization
---

## 6 — Summary Table & Statistics

In [ ]:
# ── Per-patient comparison table ──────────────────────────────────────────────
df_cmp = pd.DataFrame({'Patient': TEST_PATIENTS})
df_cmp['NoRevIN_RMSE'] = df_no_revin.RMSE.values
df_cmp['RevIN_RMSE']   = df_revin.RMSE.values
df_cmp['RMSE_Δ%']      = ((df_revin.RMSE.values - df_no_revin.RMSE.values)
                          / df_no_revin.RMSE.values * 100).round(2)
df_cmp['NoRevIN_MAE']  = df_no_revin.MAE.values
df_cmp['RevIN_MAE']    = df_revin.MAE.values
df_cmp['MAE_Δ%']       = ((df_revin.MAE.values - df_no_revin.MAE.values)
                          / df_no_revin.MAE.values * 100).round(2)
df_cmp['NoRevIN_R2']   = df_no_revin.R2.values
df_cmp['RevIN_R2']     = df_revin.R2.values
df_cmp['R2_Δ%']        = ((df_revin.R2.values - df_no_revin.R2.values)
                          / df_no_revin.R2.values * 100).round(2)

print(df_cmp.to_string(index=False))

# Save CSV
csv_path = os.path.join(OUT_DIR, f'inter_patient_revin_comparison_h{HORIZON}.csv')
df_cmp.to_csv(csv_path, index=False)
print(f'\nSaved to {csv_path}')

# Mean summary
print(f'\n{"=" * 70}')
print(f'{"Metric":>12} {"No RevIN":>12} {"+RevIN":>12} {"Δ%":>10}')
print(f'{"-" * 46}')
for metric in ['RMSE', 'MAE', 'R2']:
    nr = df_no_revin[metric].mean()
    wr = df_revin[metric].mean()
    pct = (wr - nr) / nr * 100
    print(f'{metric:>12} {nr:>12.6f} {wr:>12.6f} {pct:>+9.2f}%')
print(f'{"Time (s)":>12} {time_nr:>12.1f} {time_rv:>12.1f}')

# Wilcoxon test
if len(TEST_PATIENTS) >= 6:
    stat, p = wilcoxon(df_no_revin.RMSE.values, df_revin.RMSE.values)
    sig = '✓ Significant' if p < 0.05 else '✗ Not significant'
    print(f'\nWilcoxon signed-rank (RMSE): p={p:.4f} → {sig}')
else:
    p = float('nan')
    sig = 'N/A (< 6 patients)'
    print(f'\nWilcoxon test skipped (need ≥ 6 test patients, have {len(TEST_PATIENTS)})')

improved = (df_revin.RMSE.values < df_no_revin.RMSE.values).sum()
print(f'Patients improved: {improved}/{len(TEST_PATIENTS)}')

## 7 — Per-Patient Metric Bar Charts

In [ ]:
# ── 7.1 RMSE Bar Chart ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(12, len(TEST_PATIENTS) * 0.7), 5))
x = np.arange(len(TEST_PATIENTS))
w = 0.35

ax.bar(x - w/2, df_no_revin.RMSE.values, w,
       label='No RevIN', color='#90CAF9', edgecolor='white')
ax.bar(x + w/2, df_revin.RMSE.values, w,
       label='+ RevIN', color='#1565C0', edgecolor='white')

ax.set_xlabel('Test Patient (Unseen)', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title(f'Inter-Patient RMSE — No RevIN vs RevIN (H={HORIZON}, {len(TEST_PATIENTS)} test patients)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(TEST_PATIENTS, rotation=45, ha='right', fontsize=8)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'bar_rmse.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.2 R² Bar Chart ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(12, len(TEST_PATIENTS) * 0.7), 5))

ax.bar(x - w/2, df_no_revin.R2.values, w,
       label='No RevIN', color='#A5D6A7', edgecolor='white')
ax.bar(x + w/2, df_revin.R2.values, w,
       label='+ RevIN', color='#2E7D32', edgecolor='white')

ax.set_xlabel('Test Patient (Unseen)', fontsize=12)
ax.set_ylabel('R²', fontsize=12)
ax.set_title(f'Inter-Patient R² — No RevIN vs RevIN (H={HORIZON}, {len(TEST_PATIENTS)} test patients)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(TEST_PATIENTS, rotation=45, ha='right', fontsize=8)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'bar_r2.png'), dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.3 RMSE Improvement Waterfall ────────────────────────────────────────────
improvement = df_no_revin.RMSE.values - df_revin.RMSE.values
colors = ['#2E7D32' if imp > 0 else '#C62828' for imp in improvement]

fig, ax = plt.subplots(figsize=(max(12, len(TEST_PATIENTS) * 0.7), 5))
ax.bar(x, improvement, color=colors, edgecolor='white')
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_xlabel('Test Patient (Unseen)', fontsize=12)
ax.set_ylabel('RMSE Improvement (NoRevIN − RevIN)', fontsize=12)
ax.set_title(f'Inter-Patient RMSE Improvement (H={HORIZON}) — Green = RevIN Better', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(TEST_PATIENTS, rotation=45, ha='right', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'rmse_waterfall.png'), dpi=200, bbox_inches='tight')
plt.show()

## 8 — Signal Overlay Plots (10 Random Test Patients)

Actual vs predicted for unseen test patients — where RevIN should shine.

In [ ]:
print(f'Signal visualization patients ({len(VIZ_PATIENTS)}): {VIZ_PATIENTS}')

In [ ]:
# ── 8.1 Signal overlay — Step 1 ──────────────────────────────────────────────
for rid in VIZ_PATIENTS:
    y_true_nr, y_pred_nr = preds_no_revin[rid]
    y_true_rv, y_pred_rv = preds_revin[rid]

    N = 500
    gt      = y_true_nr[:N, 0]
    pred_nr = y_pred_nr[:N, 0]
    pred_rv = y_pred_rv[:N, 0]

    rmse_nr = np.sqrt(mean_squared_error(gt, pred_nr))
    rmse_rv = np.sqrt(mean_squared_error(gt, pred_rv))

    fig, ax = plt.subplots(figsize=(18, 4))
    ax.plot(gt, color='black', linewidth=1.2, label='Ground Truth', alpha=0.9)
    ax.plot(pred_nr, color='#E53935', linewidth=0.9,
            label=f'No RevIN (RMSE={rmse_nr:.5f})', alpha=0.7)
    ax.plot(pred_rv, color='#1565C0', linewidth=0.9,
            label=f'+ RevIN (RMSE={rmse_rv:.5f})', alpha=0.7)
    ax.set_title(f'UNSEEN Patient {rid} — Step 1 Prediction (first 500 samples)',
                 fontsize=13)
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Normalized ECG')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f'signal_overlay_{rid}_step1.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 8.2 Signal overlay — Step 10 ─────────────────────────────────────────────
for rid in VIZ_PATIENTS:
    y_true_nr, y_pred_nr = preds_no_revin[rid]
    y_true_rv, y_pred_rv = preds_revin[rid]

    N = 500
    gt      = y_true_nr[:N, -1]
    pred_nr = y_pred_nr[:N, -1]
    pred_rv = y_pred_rv[:N, -1]

    rmse_nr = np.sqrt(mean_squared_error(gt, pred_nr))
    rmse_rv = np.sqrt(mean_squared_error(gt, pred_rv))

    fig, ax = plt.subplots(figsize=(18, 4))
    ax.plot(gt, color='black', linewidth=1.2, label='Ground Truth', alpha=0.9)
    ax.plot(pred_nr, color='#E53935', linewidth=0.9,
            label=f'No RevIN (RMSE={rmse_nr:.5f})', alpha=0.7)
    ax.plot(pred_rv, color='#1565C0', linewidth=0.9,
            label=f'+ RevIN (RMSE={rmse_rv:.5f})', alpha=0.7)
    ax.set_title(f'UNSEEN Patient {rid} — Step 10 Prediction (hardest step)',
                 fontsize=13)
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Normalized ECG')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f'signal_overlay_{rid}_step10.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 8.3 Error difference — where does RevIN help most? ───────────────────────
for rid in VIZ_PATIENTS:
    y_true_nr, y_pred_nr = preds_no_revin[rid]
    y_true_rv, y_pred_rv = preds_revin[rid]

    N = 1000
    err_nr = np.abs(y_true_nr[:N, 0] - y_pred_nr[:N, 0])
    err_rv = np.abs(y_true_rv[:N, 0] - y_pred_rv[:N, 0])
    err_diff = err_nr - err_rv

    fig, axes = plt.subplots(2, 1, figsize=(18, 6), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})

    axes[0].plot(y_true_nr[:N, 0], 'k-', linewidth=0.8,
                label='Ground Truth', alpha=0.8)
    axes[0].plot(y_pred_nr[:N, 0], 'r-', linewidth=0.6,
                label='No RevIN', alpha=0.6)
    axes[0].plot(y_pred_rv[:N, 0], 'b-', linewidth=0.6,
                label='+ RevIN', alpha=0.6)
    axes[0].set_ylabel('Normalized ECG')
    axes[0].legend(fontsize=9, loc='upper right')
    axes[0].set_title(f'UNSEEN Patient {rid} — Signal + Error Difference (Step 1)',
                      fontsize=13)
    axes[0].grid(alpha=0.2)

    colors_bar = np.where(err_diff > 0, '#2E7D32', '#C62828')
    axes[1].bar(range(N), err_diff, color=colors_bar, width=1.0, alpha=0.7)
    axes[1].axhline(y=0, color='black', linewidth=0.8)
    axes[1].set_ylabel('|err_NoRevIN| − |err_RevIN|')
    axes[1].set_xlabel('Sample Index')
    axes[1].grid(alpha=0.2)

    pct_revin_wins = (err_diff > 0).mean() * 100
    axes[1].text(0.02, 0.85, f'RevIN better: {pct_revin_wins:.1f}% of samples',
                transform=axes[1].transAxes, fontsize=10,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, f'error_diff_{rid}.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

## 9 — Per-Step RMSE Comparison

In [ ]:
# ── Per-step RMSE averaged over all test patients ─────────────────────────────
step_rmse_nr = np.zeros(HORIZON)
step_rmse_rv = np.zeros(HORIZON)

for rid in TEST_PATIENTS:
    y_true_nr, y_pred_nr = preds_no_revin[rid]
    y_true_rv, y_pred_rv = preds_revin[rid]
    for h in range(HORIZON):
        step_rmse_nr[h] += np.sqrt(mean_squared_error(
            y_true_nr[:, h], y_pred_nr[:, h]))
        step_rmse_rv[h] += np.sqrt(mean_squared_error(
            y_true_rv[:, h], y_pred_rv[:, h]))

step_rmse_nr /= len(TEST_PATIENTS)
step_rmse_rv /= len(TEST_PATIENTS)

fig, ax = plt.subplots(figsize=(10, 5))
steps = range(1, HORIZON + 1)
ax.plot(steps, step_rmse_nr, 'r-o', markersize=6, linewidth=2, label='No RevIN')
ax.plot(steps, step_rmse_rv, 'b-s', markersize=6, linewidth=2, label='+ RevIN')
ax.fill_between(steps, step_rmse_nr, step_rmse_rv, alpha=0.15, color='green',
                where=(step_rmse_nr > step_rmse_rv))
ax.fill_between(steps, step_rmse_nr, step_rmse_rv, alpha=0.15, color='red',
                where=(step_rmse_nr < step_rmse_rv))
ax.set_xlabel('Forecast Step', fontsize=12)
ax.set_ylabel('Mean RMSE', fontsize=12)
ax.set_title(f'Inter-Patient Per-Step RMSE (H={HORIZON}, '
             f'avg over {len(TEST_PATIENTS)} unseen patients)', fontsize=13)
ax.set_xticks(steps)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'per_step_rmse.png'),
            dpi=200, bbox_inches='tight')
plt.show()

print(f'{"Step":>5} {"No RevIN":>10} {"+RevIN":>10} {"Δ%":>8}')
for h in range(HORIZON):
    pct = (step_rmse_rv[h] - step_rmse_nr[h]) / step_rmse_nr[h] * 100
    print(f'{h+1:>5} {step_rmse_nr[h]:>10.6f} {step_rmse_rv[h]:>10.6f} '
          f'{pct:>+7.2f}%')

## 10 — Training Curves Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, hist, title, color in [
    (axes[0], hist_nr, 'TCN (No RevIN)', '#E53935'),
    (axes[1], hist_rv, 'TCN + RevIN', '#1565C0'),
]:
    h = hist.history
    ep = range(1, len(h['loss']) + 1)
    ax.plot(ep, h['loss'], label='Train', linewidth=1.2, color=color)
    ax.plot(ep, h['val_loss'], label='Val', linewidth=1.2,
            color=color, linestyle='--', alpha=0.7)
    ax.set_title(f'{title} ({len(h["loss"])} epochs)', fontsize=12)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle(f'Inter-Patient Training Curves (H={HORIZON})', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'training_curves.png'),
            dpi=200, bbox_inches='tight')
plt.show()

## 11 — Distribution Shift Visualization

In [ ]:
# ── Show how test patients' distributions differ from train ───────────────────
# Train distribution (pooled, sample)
train_pool_sample = np.concatenate(
    [raw_signals[rid][:10000] for rid in TRAIN_PATIENTS])

n_viz_dist = min(len(VIZ_PATIENTS), 10)
nc_dist = min(3, n_viz_dist)
nr_dist = int(np.ceil(n_viz_dist / nc_dist))

fig, axes = plt.subplots(nr_dist, nc_dist, figsize=(6 * nc_dist, 4 * nr_dist))
if nr_dist == 1 and nc_dist == 1:
    axes = np.array([axes])
axes_flat = axes.flatten()

for idx, rid in enumerate(VIZ_PATIENTS[:n_viz_dist]):
    ax = axes_flat[idx]
    test_sig = raw_signals[rid]

    ax.hist(train_pool_sample, bins=100, alpha=0.5, density=True,
            label='Train (pooled)', color='#90CAF9')
    ax.hist(test_sig[:10000], bins=100, alpha=0.5, density=True,
            label=f'Test {rid}', color='#F44336')
    ax.set_title(f'Patient {rid} vs Train', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

for idx in range(n_viz_dist, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.suptitle('Inter-Patient Distribution Shift\n'
             '(why RevIN matters: test patients have different amplitude ranges)',
             fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'distribution_shift.png'),
            dpi=200, bbox_inches='tight')
plt.show()

## 12 — Scatter: No RevIN vs RevIN RMSE

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

rmse_nr_vals = df_no_revin.RMSE.values
rmse_rv_vals = df_revin.RMSE.values

ax.scatter(rmse_nr_vals, rmse_rv_vals, s=80, c='#1565C0', edgecolors='white', zorder=3)

lims = [min(rmse_nr_vals.min(), rmse_rv_vals.min()) * 0.9,
        max(rmse_nr_vals.max(), rmse_rv_vals.max()) * 1.1]
ax.plot(lims, lims, 'k--', linewidth=1, alpha=0.5, label='No change')

for i, rid in enumerate(TEST_PATIENTS):
    ax.annotate(rid, (rmse_nr_vals[i], rmse_rv_vals[i]), fontsize=7,
                textcoords='offset points', xytext=(6, 4))

ax.fill_between(lims, lims, lims[0], alpha=0.05, color='green')
ax.fill_between(lims, lims, lims[1], alpha=0.05, color='red')

ax.set_xlabel('RMSE — No RevIN', fontsize=12)
ax.set_ylabel('RMSE — + RevIN', fontsize=12)
ax.set_title(f'Inter-Patient: No RevIN vs RevIN (H={HORIZON}, {len(TEST_PATIENTS)} patients)', fontsize=13)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'scatter_rmse.png'),
            dpi=200, bbox_inches='tight')
plt.show()

## 13 — Final Verdict

In [ ]:
print('=' * 70)
print('  INTER-PATIENT TCN: No RevIN vs + RevIN')
print('=' * 70)
print(f'\n  Setup     : Train on {len(TRAIN_PATIENTS)} patients, '
      f'test on {len(TEST_PATIENTS)} UNSEEN patients')
print(f'  Total     : {len(ALL_PATIENTS)} MIT-BIH patients (all)')
print(f'  Horizon   : H={HORIZON}')
print(f'  TCN Config: blocks={NUM_BLOCKS}, filters={NUM_FILTERS}, '
      f'dropout={DROPOUT_RATE}, lr={LEARNING_RATE}')
print(f'  Source    : MOGWO Pareto-best (Eval 28)')
print()

for metric in ['RMSE', 'MAE', 'R2']:
    nr = df_no_revin[metric].mean()
    wr = df_revin[metric].mean()
    pct = (wr - nr) / nr * 100
    better = ('RevIN ✓' if (pct < 0 and metric != 'R2')
              or (pct > 0 and metric == 'R2') else 'No RevIN ✓')
    print(f'  {metric:>6}: No RevIN={nr:.6f}  +RevIN={wr:.6f}  '
          f'Δ={pct:+.2f}%  → {better}')

print(f'\n  Patients improved (RMSE): {improved}/{len(TEST_PATIENTS)}')
if not np.isnan(p):
    print(f'  Wilcoxon p-value: {p:.4f} ({sig})')
print(f'  Train time: No RevIN={time_nr:.1f}s, +RevIN={time_rv:.1f}s')
print()
print('  RevIN hypothesis for inter-patient:')
print('  RevIN normalizes each window independently → handles distribution')
print('  shift between patients without needing patient-specific scalers.')
print('  Check distribution shift plots above to see the amplitude differences.')
print()
print(f'  All outputs saved to {OUT_DIR}/')
print(f'  Plots saved to {PLOT_DIR}/')
print('=' * 70)